# Extração MediaPipe — INCLUDE-50

Este notebook faz o fluxo completo:

1. Descobre as classes presentes em `data/raw/include50`.
2. Cria/usa uma lista de 50 classes em `include50_labels.txt`.
3. Extrai landmarks com MediaPipe Holistic.
4. Salva um CSV final em `data/interim/include50/include50_mediapipe.csv`.

**Modo fácil:** se `include50_labels.txt` não existir, o notebook cria uma seleção provisória de 50 classes de forma automática e salva o arquivo.

In [10]:
from pathlib import Path
import re
import hashlib
import warnings
from typing import List, Dict, Optional

import numpy as np
import pandas as pd

# ========= CONFIGURAÇÃO PRINCIPAL =========

PROJECT_ROOT = Path.cwd().parent

RAW_ROOT = PROJECT_ROOT / "data/raw/include50"
OUTPUT_DIR = PROJECT_ROOT / "data/interim/include50"

LABELS_PATH = RAW_ROOT / "include50_labels.txt"
INVENTORY_PATH = OUTPUT_DIR / "include50_discovered_classes.csv"
SELECTED_CLASSES_PATH = OUTPUT_DIR / "include50_selected_classes.csv"
FINAL_CSV_PATH = OUTPUT_DIR / "include50_mediapipe.csv"
SHARDS_DIR = OUTPUT_DIR / "shards"

VIDEO_EXTENSIONS = {".mov", ".mp4", ".avi", ".mkv", ".MOV", ".MP4", ".AVI", ".MKV"}

# Para testar primeiro, deixe MAX_VIDEOS = 5.
# Para extrair tudo, use MAX_VIDEOS = None.
MAX_VIDEOS = None

# Se True, não reprocessa vídeo cujo shard já existe.
RESUME = True

# Se include50_labels.txt não existir, cria uma seleção provisória de 50 classes.
AUTO_CREATE_LABELS_IF_MISSING = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SHARDS_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_ROOT:", RAW_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("LABELS_PATH:", LABELS_PATH)

RAW_ROOT: /Users/dani/Projects/islr-subset/data/raw/include50
OUTPUT_DIR: /Users/dani/Projects/islr-subset/data/interim/include50
LABELS_PATH: /Users/dani/Projects/islr-subset/data/raw/include50/include50_labels.txt


In [11]:
def norm_text(value) -> str:
    value = str(value).strip().lower()
    value = re.sub(r"\s+", " ", value)
    return value

def norm_key(value) -> str:
    parts = [norm_text(p) for p in str(value).split("::")]
    return "::".join(parts)

def slugify(value: str) -> str:
    value = str(value)
    value = value.replace("/", "__").replace("\\", "__")
    value = re.sub(r"[^A-Za-z0-9_\-.]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value

def parse_class_folder(folder_name: str):
    # Exemplo: "1. loud" -> (1, "loud")
    m = re.match(r"^\s*(\d+(?:\.\d+)?)\.\s*(.+?)\s*$", folder_name)
    if not m:
        return np.nan, folder_name.strip()
    raw_id = m.group(1)
    sign = m.group(2).strip()
    try:
        original_class_id = int(float(raw_id))
    except Exception:
        original_class_id = np.nan
    return original_class_id, sign

def format_original_key(original_class_id, category, sign):
    if pd.isna(original_class_id):
        return f"{category}::{sign}"
    return f"{int(original_class_id):02d}::{category}::{sign}"

In [12]:
# Descobrir vídeos e montar metadata por vídeo

if not RAW_ROOT.exists():
    raise FileNotFoundError(f"Pasta não encontrada: {RAW_ROOT}")

video_paths = sorted([p for p in RAW_ROOT.rglob("*") if p.is_file() and p.suffix in VIDEO_EXTENSIONS])

print("Vídeos encontrados:", len(video_paths))

rows = []
bad_paths = []

for video_path in video_paths:
    rel = video_path.relative_to(RAW_ROOT)
    parts = rel.parts

    # Estrutura esperada:
    # pacote/categoria/classe/video
    # pacote/categoria/classe/Extra/video
    if len(parts) < 4:
        bad_paths.append(str(rel))
        continue

    package = parts[0]
    category = parts[1]
    class_folder = parts[2]

    original_class_id, sign = parse_class_folder(class_folder)

    class_key = f"{category}::{sign}"
    original_key = format_original_key(original_class_id, category, sign)

    # sequence_id precisa ser único mesmo se o mesmo MVI aparecer em outra categoria.
    sequence_base = str(rel.with_suffix(""))
    sequence_id = slugify(sequence_base)

    rows.append({
        "video_path": str(video_path),
        "video_relpath": str(rel),
        "package": package,
        "category": category,
        "class_folder": class_folder,
        "original_class_id": original_class_id,
        "sign": sign,
        "class_key": class_key,
        "original_key": original_key,
        "video_name": video_path.name,
        "sequence_id": sequence_id,
        "is_extra": any(norm_text(p) == "extra" for p in parts),
    })

metadata_all = pd.DataFrame(rows)

if bad_paths:
    print("Caminhos ignorados por estrutura inesperada:", len(bad_paths))
    print(bad_paths[:10])

print("Vídeos válidos no metadata:", len(metadata_all))
display(metadata_all.head())

Vídeos encontrados: 4284
Vídeos válidos no metadata: 4284


,video_path,video_relpath,package,category,class_folder,original_class_id,sign,class_key,original_key,video_name,sequence_id,is_extra
0,/Users/dani/Projects/islr-subset/data/raw/incl...,Adjectives_1of8/Adjectives/1. loud/MVI_5177.MOV,Adjectives_1of8,Adjectives,1. loud,1.0,loud,Adjectives::loud,01::Adjectives::loud,MVI_5177.MOV,Adjectives_1of8_Adjectives_1._loud_MVI_5177,False
1,/Users/dani/Projects/islr-subset/data/raw/incl...,Adjectives_1of8/Adjectives/1. loud/MVI_5178.MOV,Adjectives_1of8,Adjectives,1. loud,1.0,loud,Adjectives::loud,01::Adjectives::loud,MVI_5178.MOV,Adjectives_1of8_Adjectives_1._loud_MVI_5178,False
2,/Users/dani/Projects/islr-subset/data/raw/incl...,Adjectives_1of8/Adjectives/1. loud/MVI_5179.MOV,Adjectives_1of8,Adjectives,1. loud,1.0,loud,Adjectives::loud,01::Adjectives::loud,MVI_5179.MOV,Adjectives_1of8_Adjectives_1._loud_MVI_5179,False
3,/Users/dani/Projects/islr-subset/data/raw/incl...,Adjectives_1of8/Adjectives/1. loud/MVI_5257.MOV,Adjectives_1of8,Adjectives,1. loud,1.0,loud,Adjectives::loud,01::Adjectives::loud,MVI_5257.MOV,Adjectives_1of8_Adjectives_1._loud_MVI_5257,False
4,/Users/dani/Projects/islr-subset/data/raw/incl...,Adjectives_1of8/Adjectives/1. loud/MVI_5258.MOV,Adjectives_1of8,Adjectives,1. loud,1.0,loud,Adjectives::loud,01::Adjectives::loud,MVI_5258.MOV,Adjectives_1of8_Adjectives_1._loud_MVI_5258,False


In [13]:
# Inventário de classes disponíveis

classes_df = (
    metadata_all
    .groupby(["class_key", "original_key", "category", "sign", "original_class_id"], dropna=False)
    .agg(
        n_videos=("sequence_id", "nunique"),
        first_video=("video_relpath", "first"),
    )
    .reset_index()
    .sort_values(["category", "original_class_id", "sign"])
    .reset_index(drop=True)
)

classes_df.to_csv(INVENTORY_PATH, index=False)

print("Inventário salvo em:", INVENTORY_PATH)
print("Total de classes encontradas:", len(classes_df))
display(classes_df.head(30))

Inventário salvo em: /Users/dani/Projects/islr-subset/data/interim/include50/include50_discovered_classes.csv
Total de classes encontradas: 262


,class_key,original_key,category,sign,original_class_id,n_videos,first_video
0,Adjectives::loud,01::Adjectives::loud,Adjectives,loud,1.0,21,Adjectives_1of8/Adjectives/1. loud/MVI_5177.MOV
1,Adjectives::quiet,02::Adjectives::quiet,Adjectives,quiet,2.0,21,Adjectives_1of8/Adjectives/2. quiet/MVI_5180.MOV
2,Adjectives::happy,03::Adjectives::happy,Adjectives,happy,3.0,21,Adjectives_1of8/Adjectives/3. happy/MVI_5183.MOV
3,Adjectives::sad,04::Adjectives::sad,Adjectives,sad,4.0,8,Adjectives_1of8/Adjectives/4. sad/MVI_9565.MOV
4,Adjectives::Beautiful,05::Adjectives::Beautiful,Adjectives,Beautiful,5.0,9,Adjectives_1of8/Adjectives/5. Beautiful/Extra/...
5,Adjectives::Ugly,06::Adjectives::Ugly,Adjectives,Ugly,6.0,8,Adjectives_1of8/Adjectives/6. Ugly/MVI_9576.MOV
6,Adjectives::Deaf,07::Adjectives::Deaf,Adjectives,Deaf,7.0,8,Adjectives_1of8/Adjectives/7. Deaf/MVI_9580.MOV
7,Adjectives::Blind,08::Adjectives::Blind,Adjectives,Blind,8.0,8,Adjectives_1of8/Adjectives/8. Blind/MVI_9584.MOV
8,Adjectives::Nice,09::Adjectives::Nice,Adjectives,Nice,9.0,4,Adjectives_2of8/Adjectives/9. Nice/MVI_9588.MOV
9,Adjectives::Mean,10::Adjectives::Mean,Adjectives,Mean,10.0,8,Adjectives_2of8/Adjectives/10. Mean/MVI_9592.MOV


In [14]:
def load_labels(path: Path) -> List[str]:
    labels = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        labels.append(line)
    return labels

def resolve_labels_to_class_keys(labels: List[str], classes_df: pd.DataFrame) -> List[str]:
    classes_df = classes_df.copy()

    required_cols = ["class_key", "original_key", "sign"]
    missing = [c for c in required_cols if c not in classes_df.columns]
    if missing:
        raise ValueError(
            f"Inventário sem colunas esperadas: {missing}. "
            f"Colunas disponíveis: {classes_df.columns.tolist()}"
        )

    classes_df["norm_class_key"] = classes_df["class_key"].map(norm_key)
    classes_df["norm_original_key"] = classes_df["original_key"].map(norm_key)
    classes_df["norm_sign"] = classes_df["sign"].map(norm_text)

    resolved = []
    errors = []

    for label in labels:
        label_norm_key = norm_key(label)
        label_norm_text = norm_text(label)

        by_class_key = classes_df[classes_df["norm_class_key"] == label_norm_key]
        by_original_key = classes_df[classes_df["norm_original_key"] == label_norm_key]
        by_sign = classes_df[classes_df["norm_sign"] == label_norm_text]

        if len(by_class_key) == 1:
            resolved.append(by_class_key.iloc[0]["class_key"])
        elif len(by_original_key) == 1:
            resolved.append(by_original_key.iloc[0]["class_key"])
        elif len(by_sign) == 1:
            resolved.append(by_sign.iloc[0]["class_key"])
        elif len(by_sign) > 1:
            errors.append(
                f"Label ambíguo '{label}'. Use category::sign. Opções: "
                + ", ".join(by_sign["class_key"].tolist())
            )
        else:
            errors.append(f"Label não encontrado: {label}")

    if errors:
        raise ValueError("\n".join(errors))

    # remove duplicados mantendo ordem
    unique = []
    seen = set()
    for key in resolved:
        if key not in seen:
            unique.append(key)
            seen.add(key)

    return unique

def auto_select_50_classes(classes_df: pd.DataFrame) -> List[str]:
    # Seleção automática provisória:
    # 1) prioriza classes com mais vídeos
    # 2) faz round-robin por categoria para não pegar tudo de uma categoria só
    df = classes_df.copy()
    df = df.sort_values(["category", "n_videos", "original_class_id", "sign"], ascending=[True, False, True, True])

    selected = []
    selected_set = set()

    categories = sorted(df["category"].dropna().unique().tolist())

    while len(selected) < 50:
        added = False
        for category in categories:
            cand = df[(df["category"] == category) & (~df["class_key"].isin(selected_set))]
            if cand.empty:
                continue
            row = cand.iloc[0]
            selected.append(row["class_key"])
            selected_set.add(row["class_key"])
            added = True
            if len(selected) == 50:
                break
        if not added:
            break

    if len(selected) < 50:
        raise ValueError(f"Não foi possível selecionar 50 classes. Selecionadas: {len(selected)}")

    return selected

if not LABELS_PATH.exists():
    if not AUTO_CREATE_LABELS_IF_MISSING:
        raise FileNotFoundError(
            f"Arquivo de labels não encontrado: {LABELS_PATH}\n"
            f"Inventário salvo em: {INVENTORY_PATH}\n"
            "Crie include50_labels.txt com 50 linhas no formato Categoria::Sinal."
        )

    selected_class_keys = auto_select_50_classes(classes_df)
    LABELS_PATH.write_text("\n".join(selected_class_keys) + "\n", encoding="utf-8")
    print("ATENÇÃO: include50_labels.txt não existia.")
    print("Foi criada uma seleção PROVISÓRIA automática de 50 classes.")
    print("Arquivo criado em:", LABELS_PATH)
else:
    raw_labels = load_labels(LABELS_PATH)
    selected_class_keys = resolve_labels_to_class_keys(raw_labels, classes_df)
    print("Labels carregados:", len(raw_labels))
    print("Classes resolvidas:", len(selected_class_keys))

if len(selected_class_keys) != 50:
    raise ValueError(f"Esperava 50 classes, mas foram resolvidas {len(selected_class_keys)}.")

selected_classes = classes_df[classes_df["class_key"].isin(selected_class_keys)].copy()
selected_classes = selected_classes.set_index("class_key").loc[selected_class_keys].reset_index()
selected_classes["new_sign_id"] = range(len(selected_classes))

selected_classes.to_csv(SELECTED_CLASSES_PATH, index=False)

print("Classes selecionadas:", len(selected_classes))
print("Classes salvas em:", SELECTED_CLASSES_PATH)
display(selected_classes[["new_sign_id", "class_key", "original_key", "n_videos"]])

Labels carregados: 50
Classes resolvidas: 50
Classes selecionadas: 50
Classes salvas em: /Users/dani/Projects/islr-subset/data/interim/include50/include50_selected_classes.csv


,new_sign_id,class_key,original_key,n_videos
0,0,Adjectives::short,79::Adjectives::short,22
1,1,Animals::Bird,04::Animals::Bird,25
2,2,Clothes::Hat,37::Clothes::Hat,20
3,3,Colours::Brown,51::Colours::Brown,21
4,4,Days_and_Time::Year,78::Days_and_Time::Year,15
5,5,Electronics::Fan,53::Electronics::Fan,15
6,6,Greetings::Good afternoon,52::Greetings::Good afternoon,22
7,7,Home::Paint,40::Home::Paint,15
8,8,Jobs::Priest,91::Jobs::Priest,15
9,9,Means_of_Transportation::Train,09::Means_of_Transportation::Train,21


In [15]:
# Filtrar metadata para as 50 classes e recriar sign_id global 0..49

class_to_new_id = dict(zip(selected_classes["class_key"], selected_classes["new_sign_id"]))

metadata = metadata_all[metadata_all["class_key"].isin(selected_class_keys)].copy()
metadata["sign_id"] = metadata["class_key"].map(class_to_new_id).astype(int)
metadata["sign_key"] = metadata["sign_id"].astype(str).str.zfill(2) + "::" + metadata["category"] + "::" + metadata["sign"]

# sample_id dentro de cada classe, só para referência
metadata = metadata.sort_values(["sign_id", "video_relpath"]).reset_index(drop=True)
metadata["sample_id"] = metadata.groupby("sign_id").cumcount()

print("Vídeos selecionados:", metadata["sequence_id"].nunique())
print("Classes:", metadata["sign_id"].nunique())
print()
print(metadata.groupby(["sign_id", "class_key"])["sequence_id"].nunique().head(60).to_string())

display(metadata.head())

Vídeos selecionados: 929
Classes: 50

sign_id  class_key                     
0        Adjectives::short                 22
1        Animals::Bird                     25
2        Clothes::Hat                      20
3        Colours::Brown                    21
4        Days_and_Time::Year               15
5        Electronics::Fan                  15
6        Greetings::Good afternoon         22
7        Home::Paint                       15
8        Jobs::Priest                      15
9        Means_of_Transportation::Train    21
10       People::Brother                   21
11       Places::Court                     23
12       Pronouns::I                       21
13       Seasons::Fall                     15
14       Society::Attack                   15
15       Adjectives::small little          22
16       Animals::Cow                      21
17       Clothes::Dress                    20
18       Colours::Grey                     21
19       Days_and_Time::Second             15
20

,video_path,video_relpath,package,category,class_folder,original_class_id,sign,class_key,original_key,video_name,sequence_id,is_extra,sign_id,sign_key,sample_id
0,/Users/dani/Projects/islr-subset/data/raw/incl...,Adjectives_4of8/Adjectives/79. short/MVI_5110.MOV,Adjectives_4of8,Adjectives,79. short,79.0,short,Adjectives::short,79::Adjectives::short,MVI_5110.MOV,Adjectives_4of8_Adjectives_79._short_MVI_5110,False,0,00::Adjectives::short,0
1,/Users/dani/Projects/islr-subset/data/raw/incl...,Adjectives_4of8/Adjectives/79. short/MVI_5111.MOV,Adjectives_4of8,Adjectives,79. short,79.0,short,Adjectives::short,79::Adjectives::short,MVI_5111.MOV,Adjectives_4of8_Adjectives_79._short_MVI_5111,False,0,00::Adjectives::short,1
2,/Users/dani/Projects/islr-subset/data/raw/incl...,Adjectives_4of8/Adjectives/79. short/MVI_5112.MOV,Adjectives_4of8,Adjectives,79. short,79.0,short,Adjectives::short,79::Adjectives::short,MVI_5112.MOV,Adjectives_4of8_Adjectives_79._short_MVI_5112,False,0,00::Adjectives::short,2
3,/Users/dani/Projects/islr-subset/data/raw/incl...,Adjectives_4of8/Adjectives/79. short/MVI_5189.MOV,Adjectives_4of8,Adjectives,79. short,79.0,short,Adjectives::short,79::Adjectives::short,MVI_5189.MOV,Adjectives_4of8_Adjectives_79._short_MVI_5189,False,0,00::Adjectives::short,3
4,/Users/dani/Projects/islr-subset/data/raw/incl...,Adjectives_4of8/Adjectives/79. short/MVI_5190.MOV,Adjectives_4of8,Adjectives,79. short,79.0,short,Adjectives::short,79::Adjectives::short,MVI_5190.MOV,Adjectives_4of8_Adjectives_79._short_MVI_5190,False,0,00::Adjectives::short,4


In [16]:
# Instalação/imports da extração
# Se der erro de import mediapipe, instale no terminal:
# python -m pip install mediapipe==0.10.5 opencv-python-headless tqdm

import cv2
from tqdm.auto import tqdm
import mediapipe as mp

mp_holistic = mp.solutions.holistic

In [17]:
# Funções para extrair landmarks em formato wide

def empty_landmarks(prefix: str, n_points: int) -> Dict[str, float]:
    row = {}
    for i in range(n_points):
        row[f"{prefix}_{i}_x"] = np.nan
        row[f"{prefix}_{i}_y"] = np.nan
        row[f"{prefix}_{i}_z"] = np.nan
    return row

def add_landmarks(row: Dict[str, float], prefix: str, landmarks, n_points: int):
    if landmarks is None:
        row.update(empty_landmarks(prefix, n_points))
        return True

    pts = landmarks.landmark
    for i in range(n_points):
        if i < len(pts):
            row[f"{prefix}_{i}_x"] = pts[i].x
            row[f"{prefix}_{i}_y"] = pts[i].y
            row[f"{prefix}_{i}_z"] = pts[i].z
        else:
            row[f"{prefix}_{i}_x"] = np.nan
            row[f"{prefix}_{i}_y"] = np.nan
            row[f"{prefix}_{i}_z"] = np.nan
    return False

def shard_path_for_sequence(sequence_id: str) -> Path:
    return SHARDS_DIR / f"{sequence_id}.csv"

def extract_video_to_rows(video_meta: pd.Series, holistic) -> pd.DataFrame:
    video_path = video_meta["video_path"]
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Não foi possível abrir vídeo: {video_path}")

    rows = []
    frame_id = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = holistic.process(rgb)

        row = {
            "sign_id": int(video_meta["sign_id"]),
            "category": video_meta["category"],
            "sign": video_meta["sign"],
            "sign_key": video_meta["sign_key"],
            "sample_id": int(video_meta["sample_id"]),
            "video_name": video_meta["video_name"],
            "video_relpath": video_meta["video_relpath"],
            "sequence_id": video_meta["sequence_id"],
            "frame_id": frame_id,
        }

        missing_hand_0 = add_landmarks(row, "hand_0", result.left_hand_landmarks, 21)
        missing_hand_1 = add_landmarks(row, "hand_1", result.right_hand_landmarks, 21)
        missing_pose = add_landmarks(row, "pose", result.pose_landmarks, 33)
        missing_face = add_landmarks(row, "face", result.face_landmarks, 468)

        row["missing_hand_0"] = bool(missing_hand_0)
        row["missing_hand_1"] = bool(missing_hand_1)
        row["missing_hand"] = bool(missing_hand_0 and missing_hand_1)
        row["missing_pose"] = bool(missing_pose)
        row["missing_face"] = bool(missing_face)

        rows.append(row)
        frame_id += 1

    cap.release()

    if not rows:
        raise RuntimeError(f"Vídeo sem frames lidos: {video_path}")

    return pd.DataFrame(rows)

In [18]:
# Teste/extração
# Com MAX_VIDEOS = 5, processa só 5 vídeos para validar.
# Depois mude MAX_VIDEOS = None na primeira célula para extrair tudo.

to_process = metadata.copy()

if MAX_VIDEOS is not None:
    to_process = to_process.head(MAX_VIDEOS).copy()

print("Vídeos nesta execução:", len(to_process))
print("RESUME:", RESUME)

with mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    enable_segmentation=False,
    smooth_segmentation=False,
    refine_face_landmarks=False,
    min_detection_confidence=0.4,
    min_tracking_confidence=0.4,
) as holistic:
    for _, video_meta in tqdm(to_process.iterrows(), total=len(to_process)):
        out_path = shard_path_for_sequence(video_meta["sequence_id"])

        if RESUME and out_path.exists():
            continue

        try:
            shard_df = extract_video_to_rows(video_meta, holistic)
            shard_df.to_csv(out_path, index=False)
        except Exception as exc:
            error_path = out_path.with_suffix(".error.txt")
            error_path.write_text(str(exc), encoding="utf-8")
            print("Erro em:", video_meta["video_relpath"])
            print(exc)

print("Shards existentes:", len(list(SHARDS_DIR.glob('*.csv'))))

I0000 00:00:1782443915.604682 3485154 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro


Vídeos nesta execução: 929
RESUME: True


  0%|          | 0/929 [00:00<?, ?it/s]W0000 00:00:1782443915.677214 3488276 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782443915.687258 3488279 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782443915.689870 3488283 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782443915.689926 3488274 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782443915.689995 3488280 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782443915.692729 3488280 inference_fee

Shards existentes: 929


In [19]:
# Juntar shards em um CSV final

shard_files = sorted(SHARDS_DIR.glob("*.csv"))

if not shard_files:
    raise RuntimeError("Nenhum shard CSV encontrado.")

dfs = []
for path in tqdm(shard_files):
    dfs.append(pd.read_csv(path))

final_df = pd.concat(dfs, ignore_index=True)

# Ordenação organizada
final_df = final_df.sort_values(["sign_id", "sequence_id", "frame_id"]).reset_index(drop=True)

final_df.to_csv(FINAL_CSV_PATH, index=False)

print("CSV final salvo em:", FINAL_CSV_PATH)
print("Linhas:", len(final_df))
print("Vídeos:", final_df["sequence_id"].nunique())
print("Classes:", final_df["sign_id"].nunique())

display(final_df.head())

100%|██████████| 929/929 [00:24<00:00, 38.56it/s]


CSV final salvo em: /Users/dani/Projects/islr-subset/data/interim/include50/include50_mediapipe.csv
Linhas: 61613
Vídeos: 929
Classes: 50


,sign_id,category,sign,sign_key,sample_id,video_name,video_relpath,sequence_id,frame_id,hand_0_0_x,...,face_466_y,face_466_z,face_467_x,face_467_y,face_467_z,missing_hand_0,missing_hand_1,missing_hand,missing_pose,missing_face
0,0,Adjectives,short,00::Adjectives::short,0,MVI_5110.MOV,Adjectives_4of8/Adjectives/79. short/MVI_5110.MOV,Adjectives_4of8_Adjectives_79._short_MVI_5110,0,0.567757,...,0.236819,0.004603,0.510011,0.235687,0.004731,False,False,False,False,False
1,0,Adjectives,short,00::Adjectives::short,0,MVI_5110.MOV,Adjectives_4of8/Adjectives/79. short/MVI_5110.MOV,Adjectives_4of8_Adjectives_79._short_MVI_5110,1,0.566954,...,0.236412,0.004434,0.510173,0.235442,0.004542,False,False,False,False,False
2,0,Adjectives,short,00::Adjectives::short,0,MVI_5110.MOV,Adjectives_4of8/Adjectives/79. short/MVI_5110.MOV,Adjectives_4of8_Adjectives_79._short_MVI_5110,2,0.566794,...,0.236553,0.004450,0.510262,0.235473,0.004558,False,False,False,False,False
3,0,Adjectives,short,00::Adjectives::short,0,MVI_5110.MOV,Adjectives_4of8/Adjectives/79. short/MVI_5110.MOV,Adjectives_4of8_Adjectives_79._short_MVI_5110,3,0.566527,...,0.236887,0.004306,0.510315,0.235753,0.004415,False,False,False,False,False
4,0,Adjectives,short,00::Adjectives::short,0,MVI_5110.MOV,Adjectives_4of8/Adjectives/79. short/MVI_5110.MOV,Adjectives_4of8_Adjectives_79._short_MVI_5110,4,0.565466,...,0.236952,0.004398,0.510433,0.235741,0.004511,False,False,False,False,False


In [20]:
# Validação final

df = pd.read_csv(FINAL_CSV_PATH, usecols=[
    "sign_id", "category", "sign", "sign_key", "sample_id",
    "video_name", "video_relpath", "sequence_id", "frame_id",
    "missing_hand", "missing_hand_0", "missing_hand_1", "missing_face", "missing_pose"
])

videos = df.drop_duplicates("sequence_id")

print("Classes:", videos["sign_id"].nunique())
print("Sinais:", videos["sign_key"].nunique())
print("Vídeos:", videos["sequence_id"].nunique())
print("Frames:", len(df))

print("\nVídeos por classe:")
print(videos.groupby(["sign_id", "sign_key"])["sequence_id"].nunique().to_string())

print("\nMissing médio por frame:")
print(df[["missing_hand", "missing_hand_0", "missing_hand_1", "missing_face", "missing_pose"]].mean())

Classes: 50
Sinais: 50
Vídeos: 929
Frames: 61613

Vídeos por classe:
sign_id  sign_key                          
0        00::Adjectives::short                 22
1        01::Animals::Bird                     25
2        02::Clothes::Hat                      20
3        03::Colours::Brown                    21
4        04::Days_and_Time::Year               15
5        05::Electronics::Fan                  15
6        06::Greetings::Good afternoon         22
7        07::Home::Paint                       15
8        08::Jobs::Priest                      15
9        09::Means_of_Transportation::Train    21
10       10::People::Brother                   21
11       11::Places::Court                     23
12       12::Pronouns::I                       21
13       13::Seasons::Fall                     15
14       14::Society::Attack                   15
15       15::Adjectives::small little          22
16       16::Animals::Cow                      21
17       17::Clothes::Dress          

Para o INCLUDE-50, não foi aplicado o protocolo Leave-One-Person-Out, uma vez que a versão utilizada do dataset não disponibiliza identificadores dos sinalizadores. Seguindo a restrição metodológica observada em Alves, Boldt e Paixão, a avaliação foi conduzida por meio de uma divisão fixa em treino, validação e teste. Na ausência de um arquivo oficial de particionamento na versão obtida, os vídeos foram divididos de forma estratificada por classe, garantindo que todas as amostras de uma mesma sequência permanecessem em apenas um dos subconjuntos.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

input_csv = PROJECT_ROOT / "data/interim/include50/include50_mediapipe.csv"
output_csv = PROJECT_ROOT / "data/interim/include50/include50_mediapipe_with_split.csv"

df = pd.read_csv(input_csv)

# Uma linha por vídeo
videos = (
    df[["sequence_id", "sign_id", "sign", "sign_key"]]
    .drop_duplicates("sequence_id")
    .copy()
)

print("Vídeos:", videos["sequence_id"].nunique())
print("Classes:", videos["sign_id"].nunique())

# 15% teste
trainval, test = train_test_split(
    videos,
    test_size=0.15,
    stratify=videos["sign_id"],
    random_state=42,
)

# Dos 85% restantes, separa validação para dar 15% do total
val_size_adjusted = 0.15 / 0.85

train, val = train_test_split(
    trainval,
    test_size=val_size_adjusted,
    stratify=trainval["sign_id"],
    random_state=42,
)

split_map = {}
split_map.update({sid: "train" for sid in train["sequence_id"]})
split_map.update({sid: "val" for sid in val["sequence_id"]})
split_map.update({sid: "test" for sid in test["sequence_id"]})

df["split"] = df["sequence_id"].map(split_map)

print("\nDistribuição geral:")
print(df.drop_duplicates("sequence_id")["split"].value_counts())

print("\nDistribuição por classe:")
print(
    df.drop_duplicates("sequence_id")
      .groupby(["split", "sign_id"])
      .size()
      .groupby("split")
      .describe()
)

df.to_csv(output_csv, index=False)

print("\nSalvo em:", output_csv)

Vídeos: 929
Classes: 50

Distribuição geral:
split
train    649
val      140
test     140
Name: count, dtype: int64

Distribuição por classe:
       count   mean       std   min   25%   50%   75%   max
split                                                      
test    50.0   2.80  0.670059   2.0   2.0   3.0   3.0   4.0
train   50.0  12.98  2.368932  10.0  10.0  14.0  15.0  17.0
val     50.0   2.80  0.494872   2.0   3.0   3.0   3.0   4.0

Salvo em: /Users/dani/Projetos/islr-subset/data/interim/include50/include50_mediapipe_with_split.csv


In [2]:
import pandas as pd

path = "../data/interim/include50/include50_mediapipe_with_split.csv"
df = pd.read_csv(path, nrows=5)

print(df.columns.tolist())
print(df[["sign_id", "category", "sign", "sequence_id", "video_name", "frame_id", "split"]].head())

['sign_id', 'category', 'sign', 'sign_key', 'sample_id', 'video_name', 'video_relpath', 'sequence_id', 'frame_id', 'hand_0_0_x', 'hand_0_0_y', 'hand_0_0_z', 'hand_0_1_x', 'hand_0_1_y', 'hand_0_1_z', 'hand_0_2_x', 'hand_0_2_y', 'hand_0_2_z', 'hand_0_3_x', 'hand_0_3_y', 'hand_0_3_z', 'hand_0_4_x', 'hand_0_4_y', 'hand_0_4_z', 'hand_0_5_x', 'hand_0_5_y', 'hand_0_5_z', 'hand_0_6_x', 'hand_0_6_y', 'hand_0_6_z', 'hand_0_7_x', 'hand_0_7_y', 'hand_0_7_z', 'hand_0_8_x', 'hand_0_8_y', 'hand_0_8_z', 'hand_0_9_x', 'hand_0_9_y', 'hand_0_9_z', 'hand_0_10_x', 'hand_0_10_y', 'hand_0_10_z', 'hand_0_11_x', 'hand_0_11_y', 'hand_0_11_z', 'hand_0_12_x', 'hand_0_12_y', 'hand_0_12_z', 'hand_0_13_x', 'hand_0_13_y', 'hand_0_13_z', 'hand_0_14_x', 'hand_0_14_y', 'hand_0_14_z', 'hand_0_15_x', 'hand_0_15_y', 'hand_0_15_z', 'hand_0_16_x', 'hand_0_16_y', 'hand_0_16_z', 'hand_0_17_x', 'hand_0_17_y', 'hand_0_17_z', 'hand_0_18_x', 'hand_0_18_y', 'hand_0_18_z', 'hand_0_19_x', 'hand_0_19_y', 'hand_0_19_z', 'hand_0_20_x', 